# D330 — Apache HBase on Amazon EMR

A concise, practice-led introduction to HBase with essential operational context.

**Platform:** Amazon EMR release `emr-6.15.0`, which includes HBase `2.4.17-amzn-3`. The label `6.1.15` is not an EMR release label; use `emr-6.15.0`. Cluster creation is outside this notebook.

## Objectives

Explain when HBase fits, model a table around access patterns, distinguish HBase from Hive, describe the main components, and identify relevant EMR operational features.

> HBase is less common than relational databases, warehouses, and Spark in many modern data platforms. Focus on working knowledge rather than exhaustive administration.

## 1. The problem HBase solves

HBase is a distributed, wide-column NoSQL database built on Hadoop storage. It is designed for **low-latency random reads and writes** over very large, sparse tables.

Think of a huge key-sorted map:

```text
(row key, column family:qualifier, timestamp) -> bytes
```

Good fits include device state, time-series events, user timelines, counters, and serving a record by a known key. Poor fits include ad-hoc analytical SQL, multi-row ACID transactions, many relational joins, and small applications that a managed relational or key-value database can handle more simply.

### Thirty-second decision rule

- Need fast lookup/update by row key at Hadoop scale: consider HBase.
- Need scans, joins, grouping, and BI: use Hive/Spark/warehouse tools.
- Need both: serve operational records from HBase and analyze snapshots/exports or integrate through Hive/Spark.

## 2. HBase versus Hive

| Question | HBase | Hive |
|---|---|---|
| Main purpose | Operational NoSQL serving | SQL analytics/data warehousing |
| Typical access | `get`, `put`, bounded `scan` | Full/large scans, joins, aggregates |
| Latency target | Milliseconds for key-based access | Seconds to minutes for analytical jobs |
| Schema | Row key + column families; qualifiers can vary | Declared columns and types |
| Storage view | Mutable cells, multiple versions | Files/tables, commonly append/overwrite |
| Transactions | Atomic within one row | Depends on table format/configuration |
| Query language | Shell/client APIs; no native SQL | SQL |
| Best design starting point | Read/write access patterns | Analytical schema and partitions |

They are complementary, not competing replacements. On EMR, Hive can map an external table to HBase through `HBaseStorageHandler`, but that does not turn HBase into a warehouse. Push down row-key predicates when possible and avoid treating it as a full-scan SQL store.

## 3. Data model by example

Model the latest profile and recent orders for a customer.

```text
Table: customer_activity
Row key: CUST#1042
profile:name       = Asha
profile:city       = Chennai
metrics:order_count = 8
order:2026-08-30   = {"amount": 1250}
```

- **Table:** collection of rows, sorted lexicographically by row key.
- **Row key:** unique byte sequence; the central design decision.
- **Column family:** physical storage/retention unit declared at table creation (`profile`, `metrics`). Keep the count small.
- **Qualifier:** flexible column name inside a family (`name`, `city`). It need not exist in every row.
- **Cell:** value identified by row key, family, qualifier, and timestamp. Values are bytes; applications interpret the type.
- **Version:** another value for the same cell timestamp. Family settings control retained versions and TTL.
- **Sparse:** absent columns consume no cell storage.

A cell is not the whole row. One row may contain many qualifiers and versions.

## 4. Row-key design: performance is designed in

Rows are stored in key order and neighboring key ranges live in **regions**. A good key supports required read patterns and spreads writes across RegionServers.

| Pattern | Benefit | Risk |
|---|---|---|
| `customer_id` | Direct lookup | Cannot naturally search by city |
| `device_id#reverse_timestamp` | Recent events for one device scan together | One hot device can still hotspot |
| `salt#timestamp#id` | Distributes sequential writes | Reading a time range requires scans per salt |
| Hash of entire key | Excellent distribution | Destroys natural range scans |

Avoid globally increasing keys such as raw timestamps when ingest is heavy: all new writes target the last region. Techniques include salting/bucketing, reversing part of the key, and pre-splitting. These improve distribution but add read complexity.

### Row-key selection

For “fetch the last 20 readings for one sensor,” a practical key is `sensor_id#reverse_timestamp`. For “scan all sensors for one hour,” that key is poor unless another table or index is maintained. HBase commonly **denormalizes** and writes the same fact into multiple access-pattern tables.

## 5. How a write and read travel

```text
Client -> ZooKeeper/meta lookup -> RegionServer -> Region
                                      | write
                                      +-> WAL + MemStore -> HFiles on HDFS
                                      | background
                                      +-> flush / compaction / split
```

- **HMaster:** coordinates schema operations, region assignment, balancing, and recovery. It is not normally in every data request.
- **RegionServer:** serves reads/writes for its assigned regions.
- **Region:** contiguous row-key range of a table; regions split as they grow.
- **ZooKeeper:** coordination and discovery.
- **WAL (write-ahead log):** durable record written before an acknowledged mutation.
- **MemStore:** sorted in-memory writes per column family.
- **HFile:** immutable on-disk file in HDFS.
- **BlockCache:** caches frequently read HFile blocks.
- **Flush:** MemStore becomes an HFile.
- **Compaction:** merges HFiles and removes eligible deleted/expired/old cells. It consumes I/O.

Reads may combine MemStore, BlockCache, and multiple HFiles. Bloom filters and block indexes help avoid unnecessary disk reads.

## 6. Guarantees and important semantics

- Operations on a **single row are atomic**. A multi-column `put` for one row is seen together.
- HBase provides strong consistency for ordinary reads/writes in its standard mode.
- There is no general relational transaction spanning arbitrary rows and tables.
- Deletes write tombstone markers; space is reclaimed later during compaction.
- A scan is ordered by row key, but should not be treated as a database-wide transactional snapshot while concurrent writes occur.
- Cell timestamps can be supplied by the client; careless clock/timestamp use can make an older logical value appear newest.
- Column-family settings such as `VERSIONS`, `TTL`, compression, and Bloom filter policy affect storage and behavior.

These rules explain why row-level modeling is so important.

## 7. Feature map — what to know, not memorize

### Data access

- Point `get`, mutations (`put`, `delete`, increment, append), ordered range `scan`, filters, timestamps/versions, and batch/client APIs.
- Counters and check-and-mutate operations are useful for atomic row-level workflows.
- Filters reduce returned/processed cells but do not repair a bad row-key design.

### Organization and lifecycle

- Namespaces, tables, column families, pre-splits, TTL, version limits, compression, Bloom filters, quotas.
- Snapshots provide fast metadata-based backup/clone building blocks; exports are needed for independent/off-cluster copies.

### Scale and integration

- Automatic region splits, RegionServer balancing, bulk load through generated HFiles, MapReduce/Spark connectors, REST and Thrift gateways, and Hive mapping. Apache Phoenix (available as an EMR application) can offer SQL over HBase, with its own design and operational tradeoffs.

### Reliability, security, operations

- WAL recovery, HDFS replication, replication between HBase clusters, backup/restore tooling, metrics, logs, balancer, compaction controls, HBCK2 repair tooling, Kerberos and authorization features when configured.

“Supported” does not mean “turn everything on.” Production choices require workload tests and an operations plan.

## 8. Amazon EMR specifics

For `emr-6.15.0`, AWS documents HBase `2.4.17-amzn-3`. HBase runs on EMR on EC2 and normally stores HFiles in HDFS on core nodes. Therefore: **do not terminate the cluster expecting ordinary HDFS data to survive**. Use snapshots/exports or an appropriate persistence design before termination. Task nodes do not provide HDFS storage.

EMR 6.15.0 also introduced **Amazon EMR WAL**, an optional managed WAL capability. It must be enabled when the cluster is created, requires instance groups, and is not a substitute for backing up HFiles. Use standard HBase behavior unless the cluster was deliberately configured for EMR WAL.

Useful EMR locations and endpoints:

- HBase shell: `hbase shell` on the primary node
- HMaster UI: port `16010` (access through an SSH tunnel; do not expose it publicly)
- HBase logs: `/var/log/hbase/`
- HBase configuration: `/etc/hbase/conf/`
- HBCK2 JAR: `/usr/lib/hbase-operator-tools/` on EMR 6.1.0+; it is a repair tool, not routine maintenance

Operational rule: core-node loss, disk pressure, compaction backlog, hot regions, or undersized clusters can dominate HBase performance.

## 9. Verification questions

1. Why is `timestamp` alone usually a dangerous row key for heavy ingestion?
2. Which schema element must be declared in advance: qualifier or column family?
3. Where are acknowledged writes recorded before MemStore flush?
4. Which tool is better for a large join and aggregation: HBase or Hive?
5. Does terminating an ordinary EMR HBase cluster preserve its HDFS HFiles?

<details><summary>Answers</summary>

1. Sequential writes converge on the last key range and hotspot one region.  
2. Column family.  
3. WAL.  
4. Hive (or Spark/warehouse engine).  
5. No; plan and verify backup/export/persistence before termination.
</details>

## 10. Next notebook and references

Continue with **D331_HBase_Practical.ipynb**. It turns the model into a compact shell lab covering connection, CRUD, scans, versions, filters, counters, schema changes, snapshots, diagnostics, and cleanup.

Official references:

- [Amazon EMR 6.15.0 release](https://docs.aws.amazon.com/emr/latest/ReleaseGuide/emr-6150-release.html)
- [Apache HBase on Amazon EMR](https://docs.aws.amazon.com/emr/latest/ReleaseGuide/emr-hbase.html)
- [Using the HBase shell on EMR](https://docs.aws.amazon.com/emr/latest/ReleaseGuide/emr-hbase-connect.html)
- [Connect to the EMR primary node using SSH](https://docs.aws.amazon.com/emr/latest/ManagementGuide/emr-connect-master-node-ssh.html)
- [Apache HBase Reference Guide](https://hbase.apache.org/book.html)